# Lab: Parametric vs. Non-Parametric — Analyzing Customer Feedback

**Scenario:** As a junior data analyst at a consumer research firm, we're evaluating whether a
restaurant chain's menu redesign changed (1) customer satisfaction ratings and (2) return-visit rates.

We'll work through the four-step process: **Assess → Select → Implement → Interpret**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style="whitegrid")


## Task 1: Assessment Phase

### 1a. Load the data

In [ ]:
ratings = pd.read_csv("ratings.csv")
returns = pd.read_csv("return_visits.csv")

ratings.head()


In [ ]:
returns.head()


### 1b. Ratings — distribution, sample size, outliers

Assumes `ratings.csv` has columns `before` and `after` (rename to match your actual file if different).


In [ ]:
print("Sample size:", len(ratings))
print(ratings[["before", "after"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(ratings["before"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Ratings — Before Menu Change")
sns.histplot(ratings["after"], kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Ratings — After Menu Change")
plt.tight_layout()
plt.show()


In [ ]:
# Outlier check
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=ratings[["before", "after"]], ax=ax)
ax.set_title("Ratings — Outlier Check")
plt.show()


### 1c. Ratings — paired differences and normality check

In [ ]:
ratings["diff"] = ratings["after"] - ratings["before"]

sns.histplot(ratings["diff"], kde=True, color="seagreen")
plt.title("Distribution of Paired Differences (After - Before)")
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(ratings["diff"])
print(f"Shapiro-Wilk: statistic={shapiro_stat:.4f}, p-value={shapiro_p:.4f}")
print("Normally distributed" if shapiro_p > 0.05 else "NOT normally distributed")


### 1d. Return visits — distribution and contingency table

Assumes `return_visits.csv` has columns `location_type` (e.g. 'new_menu' / 'control') and
`returned` (e.g. 'yes' / 'no'). Adjust column names to match your actual file.


In [ ]:
contingency = pd.crosstab(returns["location_type"], returns["returned"])
print(contingency)

contingency.plot(kind="bar", stacked=True, figsize=(6, 5), colormap="Set2")
plt.title("Return Visits by Location Type")
plt.ylabel("Count")
plt.show()


In [ ]:
# Sample size and any obvious imbalance check
print(returns["location_type"].value_counts())
print(returns["returned"].value_counts())


### 1e. Equal variances check (if treating ratings as two independent groups instead of paired)

Use this only if your design calls for comparing two independent samples rather than paired
before/after values for the same customers.


In [ ]:
levene_stat, levene_p = stats.levene(ratings["before"], ratings["after"])
print(f"Levene's test: statistic={levene_stat:.4f}, p-value={levene_p:.4f}")
print("Equal variances" if levene_p > 0.05 else "Unequal variances")


---
## Task 2: Selection Phase

**Ratings (paired, continuous):**
- If the Shapiro-Wilk test above shows the differences are normally distributed (p > 0.05) and there
  are no major outliers → use a **paired t-test** (parametric).
- If the differences are NOT normally distributed, or there are significant outliers → use the
  **Wilcoxon signed-rank test** (non-parametric).

*Fill in below once you've run the checks above:*
- Normality result: ...
- Outlier assessment: ...
- **Chosen test:** ...
- **Justification:** ...

**Return visits (categorical, two independent groups):**
- This compares two categorical variables (location type vs. returned), so it isn't a
  parametric/non-parametric choice in the same sense — the appropriate test is a
  **Chi-square test of independence** on the contingency table.
- Assumption to check: expected cell counts should generally be ≥ 5 (chi2_contingency will report
  expected frequencies so you can confirm this).


## Task 3: Implementation Phase

### 3a. Ratings test

In [ ]:
# Use whichever test you selected in Task 2 based on the normality result above.

# Option A: Paired t-test (parametric)
t_stat, t_p = stats.ttest_rel(ratings["after"], ratings["before"])
print(f"Paired t-test: statistic={t_stat:.4f}, p-value={t_p:.4f}")

# Option B: Wilcoxon signed-rank test (non-parametric)
w_stat, w_p = stats.wilcoxon(ratings["diff"])
print(f"Wilcoxon signed-rank: statistic={w_stat:.4f}, p-value={w_p:.4f}")

alpha = 0.05
print("\nSignificant at alpha=0.05" if t_p < alpha else "\nNot significant at alpha=0.05")


### 3b. Return visits test

In [ ]:
chi2_stat, chi2_p, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square: statistic={chi2_stat:.4f}, p-value={chi2_p:.4f}, dof={dof}")
print("Expected frequencies:\n", expected)

print("\nSignificant association at alpha=0.05" if chi2_p < 0.05 else "\nNo significant association at alpha=0.05")


---
## Task 4: Interpretation Phase

### 4a. Cohen's d (effect size for the ratings test)


In [ ]:
cohens_d = ratings["diff"].mean() / ratings["diff"].std()
print(f"Cohen's d: {cohens_d:.4f}")

# Standard thresholds: 0.2 = small, 0.5 = medium, 0.8 = large
if abs(cohens_d) < 0.2:
    magnitude = "negligible"
elif abs(cohens_d) < 0.5:
    magnitude = "small"
elif abs(cohens_d) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"
print(f"Effect size magnitude: {magnitude}")


### 4b. Cramer's V (effect size for the return visits test)

In [ ]:
n = contingency.sum().sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2_stat / (n * min_dim))
print(f"Cramer's V: {cramers_v:.4f}")

# Standard thresholds for a 2x2 or similar small table: 0.1 = small, 0.3 = medium, 0.5 = large
if cramers_v < 0.1:
    magnitude_v = "negligible"
elif cramers_v < 0.3:
    magnitude_v = "small"
elif cramers_v < 0.5:
    magnitude_v = "medium"
else:
    magnitude_v = "large"
print(f"Effect size magnitude: {magnitude_v}")


## Conclusion

*Summarize your findings here:*
- Did the menu redesign significantly change customer ratings? How large was the effect?
- Is there a significant association between menu type and return visits? How strong is it?
- What would you recommend to the restaurant chain based on these results?
